# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Estandarizar nombres de columnas
df.columns = df.columns.str.lower().str.replace(" ", "_")
df = df.rename(columns={"monthly_premium_auto": "monthly_premium"})
df.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type'],
      dtype='str')

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [4]:
low_claim_yes = df[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")]
print(low_claim_yes.shape)
low_claim_yes.head()

(1399, 26)


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [5]:
yes_df = df[df["response"] == "Yes"]

# Prima mensual y CLV medios por policy_type y gender
premium_clv = (
    yes_df.groupby(["policy_type", "gender"])[["monthly_premium", "customer_lifetime_value"]]
    .mean()
    .round(2)
)
display(premium_clv)

# Comparación con total_claim_amount
claims = (
    yes_df.groupby(["policy_type", "gender"])["total_claim_amount"]
    .mean()
    .round(2)
)

summary = premium_clv.join(claims)
# Ratio CLV / reclamación: más alto = más rentable / menor riesgo
summary["clv_to_claim_ratio"] = (summary["customer_lifetime_value"] / summary["total_claim_amount"]).round(2)
summary["premium_to_claim_ratio"] = (summary["monthly_premium"] / summary["total_claim_amount"]).round(4)
display(summary.sort_values("clv_to_claim_ratio", ascending=False))

best = summary["clv_to_claim_ratio"].idxmax()
worst = summary["clv_to_claim_ratio"].idxmin()
print(f"Segmento más rentable / bajo riesgo: {best}")
print(f"Segmento menos rentable / mayor riesgo: {worst}")

monthly_premium  customer_lifetime_value
policy_type    gender                                          
Corporate Auto F                 94.30                  7712.63
               M                 92.19                  7944.47
Personal Auto  F                 99.00                  8339.79
               M                 91.09                  7448.38
Special Auto   F                 92.31                  7691.58
               M                 86.34                  8247.09

,,monthly_premium,customer_lifetime_value,total_claim_amount,clv_to_claim_ratio,premium_to_claim_ratio
policy_type,gender,,,,,
Corporate Auto,M,92.19,7944.47,408.58,19.44,0.2256
Special Auto,M,86.34,8247.09,429.53,19.20,0.2010
Personal Auto,F,99.00,8339.79,452.97,18.41,0.2186
Corporate Auto,F,94.30,7712.63,433.74,17.78,0.2174
Special Auto,F,92.31,7691.58,453.28,16.97,0.2036
Personal Auto,M,91.09,7448.38,457.01,16.30,0.1993


Segmento más rentable / bajo riesgo: ('Corporate Auto', 'M')
Segmento menos rentable / mayor riesgo: ('Personal Auto', 'M')


3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [6]:
customers_by_state = df.groupby("state")["customer"].count()
states_over_500 = customers_by_state[customers_by_state > 500].sort_values(ascending=False)
states_over_500

state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: customer, dtype: int64

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [7]:
clv_stats = (
    df.groupby(["education", "gender"])["customer_lifetime_value"]
    .agg(["max", "min", "median"])
    .round(2)
)
display(clv_stats)

print("Mayor CLV máximo:", clv_stats["max"].idxmax())
print("Mayor mediana de CLV:", clv_stats["median"].idxmax())
print("Menor mediana de CLV:", clv_stats["median"].idxmin())

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

Mayor CLV máximo: ('High School or Below', 'M')
Mayor mediana de CLV: ('High School or Below', 'M')
Menor mediana de CLV: ('Doctor', 'F')


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [8]:
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")
df["month"] = df["effective_to_date"].dt.month

policies_state_month = df.pivot_table(
    index="state",
    columns="month",
    values="customer",
    aggfunc="count",
    fill_value=0
)
policies_state_month

C:\Users\aleja\AppData\Local\Temp\ipykernel_19228\3200986575.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")


month,1,2
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [9]:
# 1. Agrupar por estado y mes y contar
grouped = df.groupby(["state", "month"]).size().reset_index(name="policies_sold")

# 2. Ordenar de mayor a menor
grouped = grouped.sort_values("policies_sold", ascending=False)

# 3. Top 3 estados por total de pólizas
top3_states = df["state"].value_counts().head(3).index
print("Top 3 estados:", list(top3_states))

# 4. DataFrame final: pólizas por mes de esos 3 estados
top3_by_month = (
    grouped[grouped["state"].isin(top3_states)]
    .sort_values(["state", "month"])
    .reset_index(drop=True)
)
top3_by_month

Top 3 estados: ['California', 'Oregon', 'Arizona']


,state,month,policies_sold
0,Arizona,1,1008
1,Arizona,2,929
2,California,1,1918
3,California,2,1634
4,Oregon,1,1565
5,Oregon,2,1344


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [11]:
# Unpivot de las columnas de canal/oferta
melted = df.melt(
    id_vars="response",
    value_vars=["sales_channel", "renew_offer_type"],
    var_name="channel_type",
    value_name="channel"
)

# Tasa de respuesta "Yes" (%) por canal
response_rate = (
    melted.groupby(["channel_type", "channel"])["response"]
    .apply(lambda s: (s == "Yes").mean() * 100)
    .round(2)
    .reset_index(name="response_rate_%")
    .sort_values(["channel_type", "response_rate_%"], ascending=[True, False])
)
response_rate

,channel_type,channel,response_rate_%
1,renew_offer_type,Offer2,22.03
0,renew_offer_type,Offer1,14.83
2,renew_offer_type,Offer3,1.85
3,renew_offer_type,Offer4,0.00
4,sales_channel,Agent,18.01
7,sales_channel,Web,10.89
5,sales_channel,Branch,10.79
6,sales_channel,Call Center,10.32


External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9